# 戦略2: ファクターの複合戦略（クオリティ・バリュー戦略）

**作成日**: 2026-02-21  
**目的**: 単独ファクター vs 複合ファクターのパフォーマンス比較

**ポートフォリオ**:
1. 低PBR（単独）
2. 低PER（単独）
3. 高ROE（単独）
4. 高自己資本比率（単独）
5. 高営業利益率（単独）
6. 低PBR × 高ROE（複合）
7. 低PER × 高ROE（複合）
8. 低PBR × 高自己資本比率（複合）

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

print("ライブラリインポート完了")

ライブラリインポート完了


## 1. データ読み込み・前処理

In [2]:
PROJECT_ROOT = Path(r'C:\Users\yongr\claude project\workspace')

# 価格データ
print("価格データ読み込み中...")
df_price = pd.read_parquet(PROJECT_ROOT / 'data/curated/jquants/prices/daily_quotes_all.parquet')
df_price['date'] = pd.to_datetime(df_price['date'])
df_price = df_price[df_price['date'] >= '2017-01-01'].copy()
print(f"価格データ: {len(df_price):,} 行")

# 財務データ
print("財務データ読み込み中...")
df_fin = pd.read_parquet(PROJECT_ROOT / 'data/curated/jquants/financials/statements_all.parquet')
df_fin['disclosed_date'] = pd.to_datetime(df_fin['disclosed_date'])
df_fin = df_fin[df_fin['disclosed_date'] >= '2017-01-01'].copy()

# 年次決算のみ
if 'fiscal_quarter' in df_fin.columns:
    before_count = len(df_fin)
    df_fin = df_fin[df_fin['fiscal_quarter'] == 'FY'].copy()
    print(f"年次決算フィルタ: {before_count:,} → {len(df_fin):,} 行")

print(f"財務データ: {len(df_fin):,} 行")
print("\n利用可能なカラム:")
print(df_fin.columns.tolist())

価格データ読み込み中...


価格データ: 9,148,457 行
財務データ読み込み中...


年次決算フィルタ: 171,943 → 60,649 行
財務データ: 60,649 行

利用可能なカラム:
['disclosed_date', 'disclosed_time', 'code', 'disclosure_number', 'document_type', 'fiscal_quarter', 'CurPerSt', 'CurPerEn', 'CurFYSt', 'fiscal_year_end', 'net_sales', 'operating_profit', 'ordinary_profit', 'net_profit', 'eps', 'DEPS', 'total_assets', 'equity', 'equity_ratio', 'bps', 'FSales', 'FOP', 'FOdP', 'FNP', 'FEPS', 'NxFSales', 'NxFOP', 'NxFOdP', 'NxFEPS', 'DivAnn', 'FDivFY', 'NxFDivFY', 'PayoutRatioAnn', 'ForecastNetSales', 'ForecastOperatingProfit', 'ForecastOrdinaryProfit', 'ForecastProfit', 'ForecastEarningsPerShare']


## 2. ファクター計算

In [3]:
# 必要カラムの確認と抽出
required_cols = ['disclosed_date', 'code', 'equity', 'net_profit', 'bps', 'eps', 
                 'equity_ratio', 'operating_profit', 'net_sales']

available_cols = [col for col in required_cols if col in df_fin.columns]
missing_cols = [col for col in required_cols if col not in df_fin.columns]

print("利用可能カラム:", available_cols)
print("欠損カラム:", missing_cols)

# 利用可能なカラムのみ抽出
df_fin = df_fin[available_cols].copy()

# ファクター計算
print("\nファクター計算中...")

# ROE
if 'equity' in df_fin.columns and 'net_profit' in df_fin.columns:
    df_fin['roe'] = (df_fin['net_profit'] / df_fin['equity']) * 100
    print("✓ ROE計算完了")
else:
    print("✗ ROE計算不可（equity or net_profit欠損）")

# 営業利益率
if 'operating_profit' in df_fin.columns and 'net_sales' in df_fin.columns:
    df_fin['operating_margin'] = (df_fin['operating_profit'] / df_fin['net_sales']) * 100
    print("✓ 営業利益率計算完了")
else:
    print("✗ 営業利益率計算不可（operating_profit or net_sales欠損）")

print("\nファクター統計:")
for col in ['roe', 'equity_ratio', 'operating_margin']:
    if col in df_fin.columns:
        print(f"{col}: count={df_fin[col].notna().sum():,}, mean={df_fin[col].mean():.2f}")

利用可能カラム: ['disclosed_date', 'code', 'equity', 'net_profit', 'bps', 'eps', 'equity_ratio', 'operating_profit', 'net_sales']
欠損カラム: []

ファクター計算中...
✓ ROE計算完了
✓ 営業利益率計算完了

ファクター統計:
roe: count=37,992, mean=2.78
equity_ratio: count=37,934, mean=0.52
operating_margin: count=36,904, mean=-inf


## 3. 異常値除外

In [4]:
before_count = len(df_fin)

# ROE異常値除外
if 'roe' in df_fin.columns:
    df_fin = df_fin[(df_fin['roe'] > -100) & (df_fin['roe'] < 100)].copy()

# BPS正値のみ
if 'bps' in df_fin.columns:
    df_fin = df_fin[df_fin['bps'] > 0].copy()

# EPS正値のみ（PER計算用）
if 'eps' in df_fin.columns:
    df_fin = df_fin[df_fin['eps'] > 0].copy()

# equity正値のみ
if 'equity' in df_fin.columns:
    df_fin = df_fin[df_fin['equity'] > 0].copy()

print(f"異常値除外: {before_count:,} → {len(df_fin):,} 行（{len(df_fin)/before_count*100:.1f}%）")

異常値除外: 60,649 → 32,677 行（53.9%）


## 4. 価格データのピボット化

In [5]:
print("価格データをピボット化中...")
df_price_pivot = df_price.pivot(index='date', columns='code', values='adjusted_close')
print(f"ピボットテーブル: {df_price_pivot.shape[0]} 日 × {df_price_pivot.shape[1]} 銘柄")

価格データをピボット化中...


ピボットテーブル: 2212 日 × 5226 銘柄


## 5. リバランス日生成（月次）

In [6]:
# 月末営業日
trading_days = pd.DataFrame({'date': df_price_pivot.index})
trading_days['year'] = trading_days['date'].dt.year
trading_days['month'] = trading_days['date'].dt.month
rebalance_dates = trading_days.groupby(['year', 'month'])['date'].max().values
rebalance_dates = pd.Series(rebalance_dates).sort_values().reset_index(drop=True)

print(f"リバランス日数: {len(rebalance_dates)}")
print(f"期間: {rebalance_dates.iloc[0].date()} ~ {rebalance_dates.iloc[-1].date()}")

リバランス日数: 109
期間: 2017-01-31 ~ 2026-01-22


## 6. 財務データの事前処理

In [7]:
print("財務データの事前処理中...")

fin_by_date = {}

for i, rdate in enumerate(rebalance_dates):
    if i % 12 == 0:
        print(f"  進捗: {i}/{len(rebalance_dates)}")
    
    # その日までに開示された財務データ
    available = df_fin[df_fin['disclosed_date'] <= rdate].copy()
    
    # 各銘柄の最新データ
    latest = available.sort_values('disclosed_date').groupby('code').tail(1)
    
    # インデックス設定
    latest = latest.set_index('code')
    
    fin_by_date[rdate] = latest

print("財務データ事前処理完了")

財務データの事前処理中...
  進捗: 0/109
  進捗: 12/109


  進捗: 24/109
  進捗: 36/109


  進捗: 48/109
  進捗: 60/109


  進捗: 72/109


  進捗: 84/109


  進捗: 96/109


  進捗: 108/109
財務データ事前処理完了


## 7. スクリーニング関数定義

In [8]:
def screen_low_pbr(rebalance_date, prices_pivot, fin_data, n_stocks=20):
    """低PBR戦略"""
    if rebalance_date not in prices_pivot.index or rebalance_date not in fin_data:
        return pd.DataFrame()
    
    prices = prices_pivot.loc[rebalance_date].dropna()
    fin = fin_data[rebalance_date]
    
    if 'bps' not in fin.columns:
        return pd.DataFrame()
    
    merged = pd.DataFrame({
        'adjusted_close': prices,
        'bps': fin['bps']
    }).dropna()
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    merged['pbr'] = merged['adjusted_close'] / merged['bps']
    merged = merged[(merged['pbr'] > 0.01) & (merged['pbr'] < 50)]
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # 下位25%
    pbr_q25 = merged['pbr'].quantile(0.25)
    candidates = merged[merged['pbr'] <= pbr_q25]
    
    selected = candidates.nsmallest(n_stocks, 'pbr')
    return selected.reset_index()


def screen_low_per(rebalance_date, prices_pivot, fin_data, n_stocks=20):
    """低PER戦略"""
    if rebalance_date not in prices_pivot.index or rebalance_date not in fin_data:
        return pd.DataFrame()
    
    prices = prices_pivot.loc[rebalance_date].dropna()
    fin = fin_data[rebalance_date]
    
    if 'eps' not in fin.columns:
        return pd.DataFrame()
    
    merged = pd.DataFrame({
        'adjusted_close': prices,
        'eps': fin['eps']
    }).dropna()
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    merged['per'] = merged['adjusted_close'] / merged['eps']
    merged = merged[(merged['per'] > 0) & (merged['per'] < 100)]
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # 下位25%
    per_q25 = merged['per'].quantile(0.25)
    candidates = merged[merged['per'] <= per_q25]
    
    selected = candidates.nsmallest(n_stocks, 'per')
    return selected.reset_index()


def screen_high_roe(rebalance_date, prices_pivot, fin_data, n_stocks=20):
    """高ROE戦略"""
    if rebalance_date not in prices_pivot.index or rebalance_date not in fin_data:
        return pd.DataFrame()
    
    prices = prices_pivot.loc[rebalance_date].dropna()
    fin = fin_data[rebalance_date]
    
    if 'roe' not in fin.columns:
        return pd.DataFrame()
    
    merged = pd.DataFrame({
        'adjusted_close': prices,
        'roe': fin['roe']
    }).dropna()
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # 上位25%
    roe_q75 = merged['roe'].quantile(0.75)
    candidates = merged[merged['roe'] >= roe_q75]
    
    selected = candidates.nlargest(n_stocks, 'roe')
    return selected.reset_index()


def screen_high_equity_ratio(rebalance_date, prices_pivot, fin_data, n_stocks=20):
    """高自己資本比率戦略"""
    if rebalance_date not in prices_pivot.index or rebalance_date not in fin_data:
        return pd.DataFrame()
    
    prices = prices_pivot.loc[rebalance_date].dropna()
    fin = fin_data[rebalance_date]
    
    if 'equity_ratio' not in fin.columns:
        return pd.DataFrame()
    
    merged = pd.DataFrame({
        'adjusted_close': prices,
        'equity_ratio': fin['equity_ratio']
    }).dropna()
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # 上位25%
    eq_q75 = merged['equity_ratio'].quantile(0.75)
    candidates = merged[merged['equity_ratio'] >= eq_q75]
    
    selected = candidates.nlargest(n_stocks, 'equity_ratio')
    return selected.reset_index()


def screen_high_operating_margin(rebalance_date, prices_pivot, fin_data, n_stocks=20):
    """高営業利益率戦略"""
    if rebalance_date not in prices_pivot.index or rebalance_date not in fin_data:
        return pd.DataFrame()
    
    prices = prices_pivot.loc[rebalance_date].dropna()
    fin = fin_data[rebalance_date]
    
    if 'operating_margin' not in fin.columns:
        return pd.DataFrame()
    
    merged = pd.DataFrame({
        'adjusted_close': prices,
        'operating_margin': fin['operating_margin']
    }).dropna()
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # 上位25%
    om_q75 = merged['operating_margin'].quantile(0.75)
    candidates = merged[merged['operating_margin'] >= om_q75]
    
    selected = candidates.nlargest(n_stocks, 'operating_margin')
    return selected.reset_index()


def screen_pbr_roe(rebalance_date, prices_pivot, fin_data, n_stocks=20):
    """低PBR × 高ROE複合戦略"""
    if rebalance_date not in prices_pivot.index or rebalance_date not in fin_data:
        return pd.DataFrame()
    
    prices = prices_pivot.loc[rebalance_date].dropna()
    fin = fin_data[rebalance_date]
    
    if 'bps' not in fin.columns or 'roe' not in fin.columns:
        return pd.DataFrame()
    
    merged = pd.DataFrame({
        'adjusted_close': prices,
        'bps': fin['bps'],
        'roe': fin['roe']
    }).dropna()
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    merged['pbr'] = merged['adjusted_close'] / merged['bps']
    merged = merged[(merged['pbr'] > 0.01) & (merged['pbr'] < 50)]
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # PBR下位50% かつ ROE上位50%の交差
    pbr_median = merged['pbr'].median()
    roe_median = merged['roe'].median()
    
    candidates = merged[(merged['pbr'] <= pbr_median) & (merged['roe'] >= roe_median)]
    
    if len(candidates) < n_stocks:
        # フォールバック: PBR順
        selected = merged.nsmallest(n_stocks, 'pbr')
    else:
        selected = candidates.nsmallest(n_stocks, 'pbr')
    
    return selected.reset_index()


def screen_per_roe(rebalance_date, prices_pivot, fin_data, n_stocks=20):
    """低PER × 高ROE複合戦略"""
    if rebalance_date not in prices_pivot.index or rebalance_date not in fin_data:
        return pd.DataFrame()
    
    prices = prices_pivot.loc[rebalance_date].dropna()
    fin = fin_data[rebalance_date]
    
    if 'eps' not in fin.columns or 'roe' not in fin.columns:
        return pd.DataFrame()
    
    merged = pd.DataFrame({
        'adjusted_close': prices,
        'eps': fin['eps'],
        'roe': fin['roe']
    }).dropna()
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    merged['per'] = merged['adjusted_close'] / merged['eps']
    merged = merged[(merged['per'] > 0) & (merged['per'] < 100)]
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # PER下位50% かつ ROE上位50%の交差
    per_median = merged['per'].median()
    roe_median = merged['roe'].median()
    
    candidates = merged[(merged['per'] <= per_median) & (merged['roe'] >= roe_median)]
    
    if len(candidates) < n_stocks:
        # フォールバック: PER順
        selected = merged.nsmallest(n_stocks, 'per')
    else:
        selected = candidates.nsmallest(n_stocks, 'per')
    
    return selected.reset_index()


def screen_pbr_equity(rebalance_date, prices_pivot, fin_data, n_stocks=20):
    """低PBR × 高自己資本比率複合戦略"""
    if rebalance_date not in prices_pivot.index or rebalance_date not in fin_data:
        return pd.DataFrame()
    
    prices = prices_pivot.loc[rebalance_date].dropna()
    fin = fin_data[rebalance_date]
    
    if 'bps' not in fin.columns or 'equity_ratio' not in fin.columns:
        return pd.DataFrame()
    
    merged = pd.DataFrame({
        'adjusted_close': prices,
        'bps': fin['bps'],
        'equity_ratio': fin['equity_ratio']
    }).dropna()
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    merged['pbr'] = merged['adjusted_close'] / merged['bps']
    merged = merged[(merged['pbr'] > 0.01) & (merged['pbr'] < 50)]
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # PBR下位50% かつ 自己資本比率上位50%の交差
    pbr_median = merged['pbr'].median()
    eq_median = merged['equity_ratio'].median()
    
    candidates = merged[(merged['pbr'] <= pbr_median) & (merged['equity_ratio'] >= eq_median)]
    
    if len(candidates) < n_stocks:
        # フォールバック: PBR順
        selected = merged.nsmallest(n_stocks, 'pbr')
    else:
        selected = candidates.nsmallest(n_stocks, 'pbr')
    
    return selected.reset_index()


print("スクリーニング関数定義完了")

スクリーニング関数定義完了


## 8. バックテスト関数定義

In [9]:
def run_backtest(strategy_name, screen_func, rebalance_dates, prices_pivot, fin_data, 
                 initial_cash=10_000_000, n_stocks=20, tax_rate=0.20315, unit=100, verbose=True):
    """
    バックテスト実行
    
    Returns:
        pd.DataFrame: 日次結果
    """
    cash = initial_cash
    portfolio = {}
    annual_realized_pnl = 0
    current_year = None
    results = []
    
    if verbose:
        print(f"\n{strategy_name} バックテスト開始")
    
    for i, rebalance_date in enumerate(rebalance_dates):
        if verbose and i % 12 == 0:
            print(f"  進捗: {i}/{len(rebalance_dates)} - {rebalance_date.date()}")
        
        # 年の切り替わり
        if current_year != rebalance_date.year:
            if current_year is not None and annual_realized_pnl > 0:
                tax = annual_realized_pnl * tax_rate
                cash -= tax
            annual_realized_pnl = 0
            current_year = rebalance_date.year
        
        # 既存ポートフォリオ売却
        sell_value = 0
        if rebalance_date in prices_pivot.index:
            for code, position in portfolio.items():
                if code in prices_pivot.columns:
                    sell_price = prices_pivot.loc[rebalance_date, code]
                    if pd.notna(sell_price):
                        sell_amount = position['shares'] * sell_price
                        sell_value += sell_amount
                        pnl = (sell_price - position['buy_price']) * position['shares']
                        annual_realized_pnl += pnl
        
        cash += sell_value
        portfolio = {}
        
        # 銘柄選定
        selected = screen_func(rebalance_date, prices_pivot, fin_data, n_stocks)
        
        if len(selected) == 0:
            results.append({
                'date': rebalance_date,
                'cash': cash,
                'n_stocks': 0,
                'invested': 0
            })
            continue
        
        # 購入
        target_per_stock = cash / len(selected)
        total_invested = 0
        
        for _, row in selected.iterrows():
            code = row['code']
            price = row['adjusted_close']
            shares = int(target_per_stock / (price * unit)) * unit
            
            if shares > 0:
                invest_amount = shares * price
                total_invested += invest_amount
                portfolio[code] = {'shares': shares, 'buy_price': price}
        
        cash -= total_invested
        
        results.append({
            'date': rebalance_date,
            'cash': cash,
            'n_stocks': len(portfolio),
            'invested': total_invested
        })
    
    # 最終税金
    if annual_realized_pnl > 0:
        tax = annual_realized_pnl * tax_rate
        cash -= tax
        results[-1]['cash'] = cash
    
    # 最終日の時価評価
    final_date = prices_pivot.index.max()
    final_portfolio_value = 0
    if len(portfolio) > 0:
        for code, position in portfolio.items():
            if code in prices_pivot.columns:
                final_price = prices_pivot.loc[final_date, code]
                if pd.notna(final_price):
                    final_portfolio_value += position['shares'] * final_price
        results[-1]['invested'] = final_portfolio_value
    
    if verbose:
        print(f"  完了: 最終資産 {(cash + final_portfolio_value):,.0f}円")
    
    return pd.DataFrame(results)


print("バックテスト関数定義完了")

バックテスト関数定義完了


## 9. 全戦略バックテスト実行

In [10]:
strategies = [
    ('低PBR', screen_low_pbr),
    ('低PER', screen_low_per),
    ('高ROE', screen_high_roe),
    ('高自己資本比率', screen_high_equity_ratio),
    ('高営業利益率', screen_high_operating_margin),
    ('低PBR×高ROE', screen_pbr_roe),
    ('低PER×高ROE', screen_per_roe),
    ('低PBR×高自己資本比率', screen_pbr_equity)
]

results_all = {}

print("="*60)
print("全戦略バックテスト開始")
print("="*60)

for strategy_name, screen_func in strategies:
    df_result = run_backtest(
        strategy_name=strategy_name,
        screen_func=screen_func,
        rebalance_dates=rebalance_dates,
        prices_pivot=df_price_pivot,
        fin_data=fin_by_date,
        verbose=True
    )
    results_all[strategy_name] = df_result

print("\n="*60)
print("全戦略バックテスト完了")
print("="*60)

全戦略バックテスト開始

低PBR バックテスト開始
  進捗: 0/109 - 2017-01-31


  進捗: 12/109 - 2018-01-31


  進捗: 24/109 - 2019-01-31
  進捗: 36/109 - 2020-01-31


  進捗: 48/109 - 2021-01-29


  進捗: 60/109 - 2022-01-31


...

  進捗: 60/109 - 2022-01-31


  進捗: 72/109 - 2023-01-31


  進捗: 84/109 - 2024-01-31


  進捗: 96/109 - 2025-01-31


  進捗: 108/109 - 2026-01-22
  完了: 最終資産 3,106,358円

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
全戦略バックテスト完了


## 10. パフォーマンス評価

In [11]:
def calculate_metrics(df_result, initial_cash=10_000_000):
    """
    パフォーマンス指標計算
    
    Returns:
        dict: 評価指標
    """
    df = df_result.copy()
    df['total_value'] = df['cash'] + df['invested']
    df['return'] = df['total_value'].pct_change()
    df['cumulative_return'] = (1 + df['return']).cumprod() - 1
    
    # 基本指標
    final_value = df['total_value'].iloc[-1]
    total_return = (final_value / initial_cash) - 1
    
    # 年率換算
    years = (df['date'].iloc[-1] - df['date'].iloc[0]).days / 365.25
    annual_return = (1 + total_return) ** (1 / years) - 1
    annual_volatility = df['return'].std() * np.sqrt(12)  # 月次リバランス
    
    # MDD
    df['peak'] = df['total_value'].cummax()
    df['drawdown'] = (df['total_value'] - df['peak']) / df['peak']
    mdd = df['drawdown'].min()
    
    # Sharpe, Calmar
    sharpe = (df['return'].mean() / df['return'].std() * np.sqrt(12)) if df['return'].std() > 0 else 0
    calmar = (annual_return / abs(mdd)) if mdd != 0 else 0
    
    return {
        '最終資産': final_value,
        '総リターン(%)': total_return * 100,
        '年率リターン(%)': annual_return * 100,
        '年率ボラティリティ(%)': annual_volatility * 100,
        '最大ドローダウン(%)': mdd * 100,
        'シャープレシオ': sharpe,
        'カルマー比': calmar,
        '運用期間(年)': years
    }


# 全戦略の評価指標計算
metrics_all = {}

for strategy_name, df_result in results_all.items():
    metrics = calculate_metrics(df_result)
    metrics_all[strategy_name] = metrics

# DataFrame化
df_metrics = pd.DataFrame(metrics_all).T

print("\n全戦略パフォーマンス比較")
print("="*100)
print(df_metrics.to_string())
print("="*100)


全戦略パフォーマンス比較
                      最終資産   総リターン(%)  年率リターン(%)  年率ボラティリティ(%)  最大ドローダウン(%)   シャープレシオ     カルマー比   運用期間(年)
低PBR          3.189500e+06 -68.104998 -11.955452     22.783541   -88.451433 -0.438306 -0.135164  8.974675
低PER          9.574712e+06  -4.252878  -0.483077     21.591203   -72.765254  0.088081 -0.006639  8.974675
高ROE          9.336376e+06  -6.636235  -0.762199     19.215082   -45.820459  0.059528 -0.016634  8.974675
高自己資本比率       6.780483e+06 -32.195175  -4.236883     15.393529   -52.541532 -0.203326 -0.080639  8.974675
高営業利益率        1.149857e+07  14.985688   1.568073      6.351177   -10.904174  0.275904  0.143805  8.974675
低PBR×高ROE     9.695173e+06  -3.048266  -0.344342     21.536024   -75.109661  0.093026 -0.004585  8.974675
低PER×高ROE     1.223550e+07  22.354990   2.273523     22.656122   -67.064786  0.215269  0.033900  8.974675
低PBR×高自己資本比率  3.106358e+06 -68.936423 -12.214195     20.231107   -87.254908 -0.536214 -0.139983  8.974675


## 11. ファクター比較表作成

In [12]:
# 戦略タイプ分類
single_factors = ['低PBR', '低PER', '高ROE', '高自己資本比率', '高営業利益率']
composite_factors = ['低PBR×高ROE', '低PER×高ROE', '低PBR×高自己資本比率']

# 比較表作成
comparison_data = []

for strategy in single_factors:
    if strategy in metrics_all:
        m = metrics_all[strategy]
        comparison_data.append({
            '戦略タイプ': '単独ファクター',
            '戦略名': strategy,
            '最終資産': f"{m['最終資産']:,.0f}",
            '総リターン(%)': f"{m['総リターン(%)']:.2f}",
            '年率リターン(%)': f"{m['年率リターン(%)']:.2f}",
            '年率ボラティリティ(%)': f"{m['年率ボラティリティ(%)']:.2f}",
            '最大DD(%)': f"{m['最大ドローダウン(%)']:.2f}",
            'シャープレシオ': f"{m['シャープレシオ']:.2f}",
            'カルマー比': f"{m['カルマー比']:.2f}"
        })

for strategy in composite_factors:
    if strategy in metrics_all:
        m = metrics_all[strategy]
        comparison_data.append({
            '戦略タイプ': '複合ファクター',
            '戦略名': strategy,
            '最終資産': f"{m['最終資産']:,.0f}",
            '総リターン(%)': f"{m['総リターン(%)']:.2f}",
            '年率リターン(%)': f"{m['年率リターン(%)']:.2f}",
            '年率ボラティリティ(%)': f"{m['年率ボラティリティ(%)']:.2f}",
            '最大DD(%)': f"{m['最大ドローダウン(%)']:.2f}",
            'シャープレシオ': f"{m['シャープレシオ']:.2f}",
            'カルマー比': f"{m['カルマー比']:.2f}"
        })

df_comparison = pd.DataFrame(comparison_data)

print("\n単独ファクター vs 複合ファクター 比較表")
print("="*120)
print(df_comparison.to_string(index=False))
print("="*120)


単独ファクター vs 複合ファクター 比較表
  戦略タイプ          戦略名       最終資産 総リターン(%) 年率リターン(%) 年率ボラティリティ(%) 最大DD(%) シャープレシオ カルマー比
単独ファクター         低PBR  3,189,500   -68.10    -11.96        22.78  -88.45   -0.44 -0.14
単独ファクター         低PER  9,574,712    -4.25     -0.48        21.59  -72.77    0.09 -0.01
単独ファクター         高ROE  9,336,376    -6.64     -0.76        19.22  -45.82    0.06 -0.02
単独ファクター      高自己資本比率  6,780,483   -32.20     -4.24        15.39  -52.54   -0.20 -0.08
単独ファクター       高営業利益率 11,498,569    14.99      1.57         6.35  -10.90    0.28  0.14
複合ファクター    低PBR×高ROE  9,695,173    -3.05     -0.34        21.54  -75.11    0.09 -0.00
複合ファクター    低PER×高ROE 12,235,499    22.35      2.27        22.66  -67.06    0.22  0.03
複合ファクター 低PBR×高自己資本比率  3,106,358   -68.94    -12.21        20.23  -87.25   -0.54 -0.14


## 12. 結果保存

In [13]:
output_dir = PROJECT_ROOT / 'analyses/20260221_0900_quants_model_quality_value'

# 1. backtest_results.csv（全戦略の日次パフォーマンス）
df_results_combined = pd.DataFrame()
for strategy_name, df_result in results_all.items():
    df_temp = df_result.copy()
    df_temp['total_value'] = df_temp['cash'] + df_temp['invested']
    df_temp['strategy'] = strategy_name
    df_results_combined = pd.concat([df_results_combined, df_temp[['date', 'strategy', 'total_value']]])

df_results_wide = df_results_combined.pivot(index='date', columns='strategy', values='total_value')
df_results_wide.to_csv(output_dir / 'backtest_results.csv', encoding='utf-8-sig')
print(f"✓ 保存: {output_dir / 'backtest_results.csv'}")

# 2. backtest_metrics.json
metrics_json = {}
for strategy_name, metrics in metrics_all.items():
    metrics_json[strategy_name] = {
        '最終資産': float(metrics['最終資産']),
        '総リターン(%)': float(metrics['総リターン(%)']),
        '年率リターン(%)': float(metrics['年率リターン(%)']),
        '年率ボラティリティ(%)': float(metrics['年率ボラティリティ(%)']),
        '最大ドローダウン(%)': float(metrics['最大ドローダウン(%)']),
        'シャープレシオ': float(metrics['シャープレシオ']),
        'カルマー比': float(metrics['カルマー比']),
        '運用期間(年)': float(metrics['運用期間(年)'])
    }

with open(output_dir / 'backtest_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_json, f, ensure_ascii=False, indent=2)
print(f"✓ 保存: {output_dir / 'backtest_metrics.json'}")

# 3. performance_summary.txt
with open(output_dir / 'performance_summary.txt', 'w', encoding='utf-8') as f:
    f.write("戦略2: ファクターの複合戦略（クオリティ・バリュー戦略）\n")
    f.write("="*100 + "\n\n")
    f.write(f"実行日: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"期間: {rebalance_dates.iloc[0].date()} ~ {rebalance_dates.iloc[-1].date()}\n")
    f.write(f"初期資本: 10,000,000円\n")
    f.write(f"リバランス: 月次（{len(rebalance_dates)}回）\n\n")
    f.write("="*100 + "\n")
    f.write("全戦略パフォーマンス比較\n")
    f.write("="*100 + "\n")
    f.write(df_metrics.to_string())
    f.write("\n\n" + "="*100 + "\n")
    f.write("単独ファクター vs 複合ファクター 比較\n")
    f.write("="*100 + "\n")
    f.write(df_comparison.to_string(index=False))
    f.write("\n")

print(f"✓ 保存: {output_dir / 'performance_summary.txt'}")

# 4. factor_comparison.csv
df_comparison.to_csv(output_dir / 'factor_comparison.csv', index=False, encoding='utf-8-sig')
print(f"✓ 保存: {output_dir / 'factor_comparison.csv'}")

print("\n全ての成果物を保存完了")

✓ 保存: C:\Users\yongr\claude project\workspace\analyses\20260221_0900_quants_model_quality_value\backtest_results.csv
✓ 保存: C:\Users\yongr\claude project\workspace\analyses\20260221_0900_quants_model_quality_value\backtest_metrics.json
✓ 保存: C:\Users\yongr\claude project\workspace\analyses\20260221_0900_quants_model_quality_value\performance_summary.txt
✓ 保存: C:\Users\yongr\claude project\workspace\analyses\20260221_0900_quants_model_quality_value\factor_comparison.csv

全ての成果物を保存完了


## 13. ランキング分析

In [14]:
# 各指標でランキング
print("\n指標別ランキング")
print("="*80)

ranking_metrics = ['総リターン(%)', '年率リターン(%)', 'シャープレシオ', 'カルマー比']

for metric in ranking_metrics:
    if metric in df_metrics.columns:
        print(f"\n【{metric}】")
        top3 = df_metrics.nlargest(3, metric)[[metric]]
        for i, (strategy, value) in enumerate(top3.iterrows(), 1):
            print(f"  {i}位: {strategy} ({value[metric]:.2f})")

print("\n最大ドローダウン（低い方が良い）")
top3_dd = df_metrics.nsmallest(3, '最大ドローダウン(%)')[['最大ドローダウン(%)']]
for i, (strategy, value) in enumerate(top3_dd.iterrows(), 1):
    print(f"  {i}位: {strategy} ({value['最大ドローダウン(%)']:.2f}%)")

print("="*80)


指標別ランキング

【総リターン(%)】
  1位: 低PER×高ROE (22.35)
  2位: 高営業利益率 (14.99)
  3位: 低PBR×高ROE (-3.05)

【年率リターン(%)】
  1位: 低PER×高ROE (2.27)
  2位: 高営業利益率 (1.57)
  3位: 低PBR×高ROE (-0.34)

【シャープレシオ】
  1位: 高営業利益率 (0.28)
  2位: 低PER×高ROE (0.22)
  3位: 低PBR×高ROE (0.09)

【カルマー比】
  1位: 高営業利益率 (0.14)
  2位: 低PER×高ROE (0.03)
  3位: 低PBR×高ROE (-0.00)

最大ドローダウン（低い方が良い）
  1位: 低PBR (-88.45%)
  2位: 低PBR×高自己資本比率 (-87.25%)
  3位: 低PBR×高ROE (-75.11%)


## 完了